# P4 — Time-Series Foundation Models

## Mục tiêu

Notebook triển khai và so sánh các time-series foundation models trên mẫu M5:

- **Chronos-Bolt** — zero-shot, dự báo quantile native.
- **TinyTimeMixer (TTM / IBM Granite)** — zero-shot, dự báo điểm và chuyển sang dự báo xác suất.
- **Moirai-2** — zero-shot, dự báo xác suất native.
- Đo **runtime** của các foundation models trên cùng giao thức.
- Phân tích nguy cơ **pre-training leakage / contamination** khi đánh giá zero-shot trên M5.
- **Few-shot ablation** cho TTM và Moirai nếu đủ tài nguyên/thời gian.

Zero-shot là kết quả chính của nhóm Foundation Models; few-shot chỉ được xem là phân tích bổ sung.

## Giao thức thực nghiệm

- Dữ liệu: mẫu phân tầng từ M5.
- Không shuffle theo thời gian.
- Horizon: 28 ngày.
- Sử dụng các rolling-origin windows do P1 cung cấp.
- Mỗi model chỉ sử dụng dữ liệu đến `train_end` tại mỗi origin.
- Validation và test không được đưa vào context đầu vào.
- Output chuẩn:
  `series_id, date, d_int, quantile_level, value, model`.

In [1]:
# ============================================================
# 01 — ENVIRONMENT & REPRODUCIBILITY
# ============================================================

from pathlib import Path
import platform
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import yaml
import torch
from scipy import stats

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Python :", sys.version.split()[0])
print("System :", platform.platform())
print("PyTorch:", torch.__version__)
print("Device :", DEVICE)
print("Python executable:", sys.executable)

Python : 3.12.6
System : macOS-15.6.1-arm64-arm-64bit
PyTorch: 2.11.0
Device : mps
Python executable: /Users/ntt2711/m5-intermittent-forecasting/.venv/bin/python


In [3]:
# ============================================================
# 02 — CONFIGURATION & INPUT VALIDATION
# ============================================================

CWD = Path.cwd()

if (CWD / "common" / "config.yaml").exists():
    ROOT = CWD
elif (CWD.parent / "common" / "config.yaml").exists():
    ROOT = CWD.parent
else:
    raise FileNotFoundError(
        f"Không xác định được project root từ: {CWD}"
    )

print("Working directory:", CWD)
print("Project root     :", ROOT)

PATH_CONFIG = ROOT / "common/config.yaml"
PATH_WINDOWS = ROOT / "common/rolling_windows.csv"
PATH_SAMPLE = ROOT / "common/sample_ids.csv"

PATH_CALENDAR = ROOT / "data/raw/calendar.csv"
PATH_LONG = ROOT / "data/processed/data_long.parquet"

PATH_OUTPUT = ROOT / "data/outputs/forecasts"
PATH_OUTPUT.mkdir(parents=True, exist_ok=True)

required_files = {
    "config": PATH_CONFIG,
    "rolling windows": PATH_WINDOWS,
    "sample ids": PATH_SAMPLE,
    "calendar": PATH_CALENDAR,
    "data_long": PATH_LONG,
}

for name, path in required_files.items():
    assert path.exists(), f"Thiếu {name}: {path}"
    print(f"{name:<16} {path}")

Working directory: /Users/ntt2711/m5-intermittent-forecasting/notebooks
Project root     : /Users/ntt2711/m5-intermittent-forecasting
config           /Users/ntt2711/m5-intermittent-forecasting/common/config.yaml
rolling windows  /Users/ntt2711/m5-intermittent-forecasting/common/rolling_windows.csv
sample ids       /Users/ntt2711/m5-intermittent-forecasting/common/sample_ids.csv
calendar         /Users/ntt2711/m5-intermittent-forecasting/data/raw/calendar.csv
data_long        /Users/ntt2711/m5-intermittent-forecasting/data/processed/data_long.parquet


In [4]:
with open(PATH_CONFIG, encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

Q = [float(q) for q in CFG["quantiles"]]
H = int(CFG["horizon"])
SP = CFG["split"]
FM = CFG["foundation"]

windows = pd.read_csv(PATH_WINDOWS)
sample_ids = pd.read_csv(PATH_SAMPLE)
calendar = pd.read_csv(PATH_CALENDAR)
long = pd.read_parquet(PATH_LONG)

print("\nConfiguration")
print("Quantiles :", Q)
print("Horizon   :", H)
print("Windows   :", len(windows))

print("\nData")
print("data_long :", long.shape)
print("Sample IDs:", sample_ids.shape)
print("Calendar  :", calendar.shape)

display(windows)


Configuration
Quantiles : [0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]
Horizon   : 28
Windows   : 3

Data
data_long : (6743034, 24)
Sample IDs: (3474, 1)
Calendar  : (1969, 14)


,window,train_end,val_end,test_start,test_end
0,W1,1885,1913,1914,1941
1,W2,1857,1885,1886,1913
2,W3,1829,1857,1858,1885


In [5]:
# ============================================================
# 03 — PREPARE FOUNDATION-MODEL DATA
# ============================================================

REQUIRED_COLS = {
    "series_id",
    "date",
    "d_int",
    "sales",
    "segment",
}

missing = REQUIRED_COLS - set(long.columns)

assert not missing, (
    f"data_long thiếu các cột: {sorted(missing)}"
)

p4_data = long[
    [
        "series_id",
        "date",
        "d_int",
        "sales",
        "segment",
    ]
].copy()

p4_data["date"] = pd.to_datetime(
    p4_data["date"]
)

p4_data["d_int"] = pd.to_numeric(
    p4_data["d_int"]
).astype(int)

p4_data["sales"] = pd.to_numeric(
    p4_data["sales"]
).astype(np.float32)

p4_data = (
    p4_data
    .sort_values(["series_id", "d_int"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Validate sample
# ------------------------------------------------------------

expected_ids = set(
    sample_ids["series_id"].astype(str)
)

actual_ids = set(
    p4_data["series_id"].astype(str)
)

assert actual_ids == expected_ids, (
    "series_id trong data_long không khớp sample_ids.csv"
)

# Mỗi d_int tương ứng đúng một date
date_map_check = (
    p4_data
    .groupby("d_int")["date"]
    .nunique()
)

assert date_map_check.max() == 1

day_to_date = (
    p4_data[
        ["d_int", "date"]
    ]
    .drop_duplicates()
    .set_index("d_int")["date"]
    .to_dict()
)

N_SERIES = p4_data["series_id"].nunique()

COLS = [
    "series_id",
    "date",
    "d_int",
    "quantile_level",
    "value",
    "model",
]

print("P4 data ready")
print("Series :", f"{N_SERIES:,}")
print(
    "Days   :",
    p4_data["d_int"].min(),
    "→",
    p4_data["d_int"].max()
)
print(
    "Dates  :",
    p4_data["date"].min().date(),
    "→",
    p4_data["date"].max().date()
)

print("\nDemand segments:")
display(
    p4_data[
        ["series_id", "segment"]
    ]
    .drop_duplicates()["segment"]
    .value_counts()
)

P4 data ready
Series : 3,474
Days   : 1 → 1941
Dates  : 2011-01-29 → 2016-05-22

Demand segments:


segment
lumpy           1000
intermittent    1000
smooth           980
erratic          494
Name: count, dtype: int64

In [6]:
# ============================================================
# 04 — COMMON VALIDATION HELPERS
# ============================================================

def validate_forecast(df, model_name):
    """Kiểm tra forecast theo schema chuẩn của project."""

    assert list(df.columns) == COLS, (
        f"{model_name}: sai schema.\n"
        f"Expected: {COLS}\n"
        f"Actual:   {list(df.columns)}"
    )

    expected_rows = (
        len(windows)
        * N_SERIES
        * H
        * len(Q)
    )

    assert len(df) == expected_rows, (
        f"{model_name}: {len(df):,} rows, "
        f"expected {expected_rows:,}"
    )

    assert df["series_id"].nunique() == N_SERIES
    assert df["quantile_level"].nunique() == len(Q)

    assert not df["value"].isna().any()
    assert np.isfinite(df["value"]).all()
    assert (df["value"] >= 0).all()

    assert set(df["model"].unique()) == {model_name}

    print(
        f"✓ {model_name}: "
        f"{len(df):,} rows | "
        f"{N_SERIES:,} series | "
        f"{len(Q)} quantiles"
    )


def check_quantile_crossing(df, model_name):
    """Kiểm tra q thấp <= q cao tại mỗi series × ngày."""

    wide = df.pivot_table(
        index=["series_id", "d_int"],
        columns="quantile_level",
        values="value",
    )

    arr = wide[sorted(Q)].to_numpy()

    crossing = np.any(
        np.diff(arr, axis=1) < 0,
        axis=1,
    )

    n_crossing = int(crossing.sum())

    print(
        f"✓ {model_name}: "
        f"quantile crossing = {n_crossing:,}"
    )

    assert n_crossing == 0

## Chronos-Bolt: Zero-shot

Chronos-Bolt được sử dụng ở chế độ **zero-shot**: model pretrained được dùng trực tiếp,
không fine-tune trên M5.

Tại mỗi rolling origin, model chỉ nhận tối đa 512 quan sát trước hoặc tại `train_end`.
Để tránh sử dụng validation như dữ liệu đầu vào, model dự báo liên tục từ `train_end + 1`
đến `test_end`; sau đó chỉ giữ lại 28 ngày thuộc test window.

Chronos-Bolt cung cấp dự báo quantile native. Tuy nhiên, model chỉ được huấn luyện cho
các mức quantile trong khoảng `[0.1, 0.9]`. Các quantile cực biên của giao thức M5 nằm
ngoài khoảng này vì vậy bị giới hạn tại biên hỗ trợ của model. Forecast tại đây được
giữ ở dạng **raw/uncalibrated**; bước calibration thuộc giai đoạn đánh giá chung.

In [7]:
from chronos import BaseChronosPipeline

CHRONOS_MODEL = FM["chronos_model"]
CHRONOS_CONTEXT = int(FM["chronos_context"])

print("Model  :", CHRONOS_MODEL)
print("Context:", CHRONOS_CONTEXT)

/Users/ntt2711/m5-intermittent-forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model  : amazon/chronos-bolt-base
Context: 512


In [9]:
chronos_device = "mps"

chronos = BaseChronosPipeline.from_pretrained(
    CHRONOS_MODEL,
    device_map=chronos_device,
    torch_dtype=torch.float32,
)

print("Chronos-Bolt loaded on", chronos_device)

Loading weights: 100%|██████████| 269/269 [00:00<00:00, 1462.88it/s]


Chronos-Bolt loaded on mps


In [10]:
# ============================================================
# CHRONOS SMOKE TEST
# ============================================================

w = windows.iloc[0]

train_end = int(w["train_end"])
test_start = int(w["test_start"])
test_end = int(w["test_end"])

# Forecast train_end+1 → test_end.
# Validation tuyệt đối không được đưa vào context.
total_h = test_end - train_end

test_ids = (
    p4_data["series_id"]
    .drop_duplicates()
    .iloc[:5]
    .tolist()
)

contexts = []

for sid in test_ids:

    y = (
        p4_data[
            (p4_data["series_id"] == sid)
            & (p4_data["d_int"] <= train_end)
        ]
        .sort_values("d_int")["sales"]
        .to_numpy(dtype=np.float32)
    )

    y = y[-CHRONOS_CONTEXT:]

    contexts.append(
        torch.tensor(y, dtype=torch.float32)
    )

context = torch.stack(contexts)

print("Input shape :", context.shape)
print("Forecast h  :", total_h)

with torch.no_grad():

    q_test, mean_test = chronos.predict_quantiles(
        inputs=context,
        prediction_length=total_h,
        quantile_levels=Q,
    )

print("Quantile shape:", tuple(q_test.shape))
print("Mean shape    :", tuple(mean_test.shape))

Input shape : torch.Size([5, 512])
Forecast h  : 56


	Quantiles to be predicted ([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]) are not within the range of quantiles that Chronos-Bolt was trained on ([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]). Quantile predictions will be set to the minimum/maximum levels at which Chronos-Bolt was trained on. This may significantly affect the quality of the predictions.


Quantile shape: (5, 56, 9)
Mean shape    : (5, 56)


In [11]:
# ============================================================
# CHRONOS FULL ROLLING INFERENCE
# ============================================================

CHRONOS_BATCH_SIZE = 64

chronos_outputs = []
chronos_runtime = []

series_ids = (
    p4_data["series_id"]
    .drop_duplicates()
    .tolist()
)

print(f"Series     : {len(series_ids):,}")
print(f"Windows    : {len(windows)}")
print(f"Batch size : {CHRONOS_BATCH_SIZE}")
print(f"Device     : {chronos_device}")

Series     : 3,474
Windows    : 3
Batch size : 64
Device     : mps


In [12]:
for w_idx, w in windows.iterrows():

    window_name = (
        str(w["window"])
        if "window" in windows.columns
        else f"W{w_idx + 1}"
    )

    train_end = int(w["train_end"])
    test_start = int(w["test_start"])
    test_end = int(w["test_end"])

    prediction_length = test_end - train_end

    print(
        f"\n{'=' * 60}\n"
        f"{window_name} | "
        f"train_end={train_end} | "
        f"test={test_start}–{test_end} | "
        f"forecast_length={prediction_length}"
    )

    window_start = time.perf_counter()
    window_parts = []

    for batch_start in range(
        0,
        len(series_ids),
        CHRONOS_BATCH_SIZE,
    ):

        batch_ids = series_ids[
            batch_start:
            batch_start + CHRONOS_BATCH_SIZE
        ]

        contexts = []

        for sid in batch_ids:

            y = (
                p4_data.loc[
                    (p4_data["series_id"] == sid)
                    & (p4_data["d_int"] <= train_end),
                    ["d_int", "sales"],
                ]
                .sort_values("d_int")["sales"]
                .to_numpy(dtype=np.float32)
            )

            y = y[-CHRONOS_CONTEXT:]

            contexts.append(
                torch.tensor(
                    y,
                    dtype=torch.float32,
                )
            )

        context_batch = torch.stack(contexts)

        # ----------------------------------------------------
        # Zero-shot inference
        # ----------------------------------------------------

        with torch.inference_mode():

            q_pred, _ = chronos.predict_quantiles(
                inputs=context_batch,
                prediction_length=prediction_length,
                quantile_levels=Q,
            )

        q_pred = (
            q_pred
            .detach()
            .cpu()
            .numpy()
        )

        # ----------------------------------------------------
        # Keep TEST horizon only
        #
        # Forecast starts at train_end + 1.
        # Validation occupies the first part of the forecast.
        # ----------------------------------------------------

        test_offset = test_start - (train_end + 1)

        q_test = q_pred[
            :,
            test_offset:test_offset + H,
            :,
        ]

        assert q_test.shape == (
            len(batch_ids),
            H,
            len(Q),
        )

        # Demand cannot be negative
        q_test = np.clip(q_test, 0, None)

        # ----------------------------------------------------
        # Long-format output
        # ----------------------------------------------------

        test_days = np.arange(
            test_start,
            test_end + 1,
        )

        for i, sid in enumerate(batch_ids):

            block = pd.DataFrame({
                "series_id": np.repeat(
                    sid,
                    H * len(Q),
                ),

                "d_int": np.repeat(
                    test_days,
                    len(Q),
                ),

                "quantile_level": np.tile(
                    Q,
                    H,
                ),

                "value": q_test[i].reshape(-1),
            })

            block["date"] = block["d_int"].map(
                day_to_date
            )

            block["model"] = "chronos_bolt"

            window_parts.append(
                block[COLS]
            )

        done = min(
            batch_start + CHRONOS_BATCH_SIZE,
            len(series_ids),
        )

        if (
            done % (CHRONOS_BATCH_SIZE * 10) == 0
            or done == len(series_ids)
        ):
            print(
                f"  {done:,}/{len(series_ids):,} series"
            )

    # --------------------------------------------------------
    # Finish window
    # --------------------------------------------------------

    window_df = pd.concat(
        window_parts,
        ignore_index=True,
    )

    elapsed = time.perf_counter() - window_start

    chronos_runtime.append({
        "model": "chronos_bolt",
        "window": window_name,
        "n_series": len(series_ids),
        "prediction_length": prediction_length,
        "seconds": elapsed,
    })

    chronos_outputs.append(window_df)

    # Checkpoint
    checkpoint_path = (
        PATH_OUTPUT /
        f"chronos_bolt_{window_name}.parquet"
    )

    window_df.to_parquet(
        checkpoint_path,
        index=False,
    )

    print(
        f"✓ {window_name}: "
        f"{len(window_df):,} rows | "
        f"{elapsed:.1f}s"
    )

    # Release temporary memory
    del window_parts, window_df


W1 | train_end=1885 | test=1914–1941 | forecast_length=56


	Quantiles to be predicted ([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]) are not within the range of quantiles that Chronos-Bolt was trained on ([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]). Quantile predictions will be set to the minimum/maximum levels at which Chronos-Bolt was trained on. This may significantly affect the quality of the predictions.
	Quantiles to be predicted ([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]) are not within the range of quantiles that Chronos-Bolt was trained on ([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]). Quantile predictions will be set to the minimum/maximum levels at which Chronos-Bolt was trained on. This may significantly affect the quality of the predictions.
	Quantiles to be predicted ([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]) are not within the range of quantiles that Chronos-Bolt was trained on ([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]). Quantile predictions will be set to the minimum

  640/3,474 series


	Quantiles to be predicted ([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]) are not within the range of quantiles that Chronos-Bolt was trained on ([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]). Quantile predictions will be set to the minimum/maximum levels at which Chronos-Bolt was trained on. This may significantly affect the quality of the predictions.
	Quantiles to be predicted ([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]) are not within the range of quantiles that Chronos-Bolt was trained on ([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]). Quantile predictions will be set to the minimum/maximum levels at which Chronos-Bolt was trained on. This may significantly affect the quality of the predictions.
	Quantiles to be predicted ([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]) are not within the range of quantiles that Chronos-Bolt was trained on ([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]). Quantile predictions will be set to the minimum

  1,280/3,474 series


...

  2,560/3,474 series


	Quantiles to be predicted ([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]) are not within the range of quantiles that Chronos-Bolt was trained on ([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]). Quantile predictions will be set to the minimum/maximum levels at which Chronos-Bolt was trained on. This may significantly affect the quality of the predictions.
	Quantiles to be predicted ([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]) are not within the range of quantiles that Chronos-Bolt was trained on ([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]). Quantile predictions will be set to the minimum/maximum levels at which Chronos-Bolt was trained on. This may significantly affect the quality of the predictions.
	Quantiles to be predicted ([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]) are not within the range of quantiles that Chronos-Bolt was trained on ([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]). Quantile predictions will be set to the minimum

  3,200/3,474 series


	Quantiles to be predicted ([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]) are not within the range of quantiles that Chronos-Bolt was trained on ([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]). Quantile predictions will be set to the minimum/maximum levels at which Chronos-Bolt was trained on. This may significantly affect the quality of the predictions.
	Quantiles to be predicted ([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]) are not within the range of quantiles that Chronos-Bolt was trained on ([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]). Quantile predictions will be set to the minimum/maximum levels at which Chronos-Bolt was trained on. This may significantly affect the quality of the predictions.
	Quantiles to be predicted ([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995]) are not within the range of quantiles that Chronos-Bolt was trained on ([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]). Quantile predictions will be set to the minimum

  3,474/3,474 series
✓ W3: 875,448 rows | 915.3s


In [13]:
# ============================================================
# COMBINE & VALIDATE CHRONOS FORECAST
# ============================================================

chronos_all = pd.concat(
    chronos_outputs,
    ignore_index=True,
)

chronos_all["quantile_level"] = (
    chronos_all["quantile_level"]
    .astype(float)
)

chronos_all["value"] = (
    chronos_all["value"]
    .astype(np.float32)
)

validate_forecast(
    chronos_all,
    "chronos_bolt",
)

check_quantile_crossing(
    chronos_all,
    "chronos_bolt",
)

display(
    pd.DataFrame(chronos_runtime)
)

✓ chronos_bolt: 2,626,344 rows | 3,474 series | 9 quantiles
✓ chronos_bolt: quantile crossing = 0


,model,window,n_series,prediction_length,seconds
0,chronos_bolt,W1,3474,56,973.269981
1,chronos_bolt,W2,3474,56,937.250327
2,chronos_bolt,W3,3474,56,915.324271


In [14]:
example_sid = series_ids[0]
example_day = int(windows.iloc[0]["test_start"])

example_q = (
    chronos_all[
        (chronos_all["series_id"] == example_sid)
        & (chronos_all["d_int"] == example_day)
    ]
    [["quantile_level", "value"]]
    .sort_values("quantile_level")
)

display(example_q)

,quantile_level,value
0,0.005,0.000000
1,0.025,0.000000
2,0.165,0.000000
3,0.250,0.010286
4,0.500,0.189975
5,0.750,1.570977
6,0.835,2.465563
7,0.975,3.279027
8,0.995,3.279027


In [17]:
# ============================================================
# SAVE FINAL CHRONOS OUTPUT
# ============================================================

chronos_path = PATH_OUTPUT / "chronos_bolt.parquet"

assert list(chronos_all.columns) == COLS, (
    f"Invalid output schema: {list(chronos_all.columns)}"
)

chronos_all[COLS].to_parquet(
    chronos_path,
    index=False,
)

print(
    f"  Saved Chronos-Bolt forecast\n"
    f"  Path : {chronos_path}\n"
    f"  Shape: {chronos_all.shape}\n"
    f"  Size : {chronos_path.stat().st_size / 1024**2:.1f} MB"
)

  Saved Chronos-Bolt forecast
  Path : /Users/ntt2711/m5-intermittent-forecasting/data/outputs/forecasts/chronos_bolt.parquet
  Shape: (2626344, 6)
  Size : 10.1 MB


## TinyTimeMixer (TTM): Zero-shot

TinyTimeMixer (TTM) được sử dụng ở chế độ **zero-shot**, không fine-tune trên dữ liệu M5.

Tại mỗi rolling origin:

1. Chỉ các quan sát `d_int <= train_end` được sử dụng làm context.
2. Tối đa 512 quan sát gần nhất được đưa vào model.
3. Mỗi chuỗi được chuẩn hóa bằng mean và standard deviation tính **chỉ trên context**.
4. TTM tạo point forecast từ `train_end + 1` đến `test_end`.
5. Chỉ 28 ngày thuộc test window được giữ lại.
6. Point forecasts được chuyển thành 9 quantiles bằng Negative Binomial hoặc Poisson wrapper.

Việc ước lượng các tham số của probabilistic wrapper cũng chỉ sử dụng dữ liệu
trước hoặc tại `train_end`, nhằm tránh leakage từ validation/test.

In [18]:
# ============================================================
# LOAD TINYTIMEMIXER
# ============================================================

from tsfm_public.models.tinytimemixer import (
    TinyTimeMixerForPrediction,
)

TTM_MODEL = FM["ttm_model"]
TTM_CONTEXT = int(FM["ttm_context"])

print("Model  :", TTM_MODEL)
print("Context:", TTM_CONTEXT)
print("Device :", DEVICE)

Model  : ibm-granite/granite-timeseries-ttm-r2
Context: 512
Device : mps


In [19]:
ttm = TinyTimeMixerForPrediction.from_pretrained(
    TTM_MODEL,
    num_input_channels=1,
)

ttm = ttm.to(DEVICE)
ttm.eval()

print("TTM loaded")
print("Context length   :", ttm.config.context_length)
print("Prediction length:", ttm.config.prediction_length)

INFO:p-85697:t-8310546624:_client.py:_send_single_request:HTTP Request: HEAD https://huggingface.co/ibm-granite/granite-timeseries-ttm-r2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:p-85697:t-8310546624:_client.py:_send_single_request:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ibm-granite/granite-timeseries-ttm-r2/d6a79570cac0f33d526601cd3a0fc7c80a8f9a2f/config.json?%2Fibm-granite%2Fgranite-timeseries-ttm-r2%2Fresolve%2Fmain%2Fconfig.json=&etag=%2209e937ef5082573c0785a180f23753110b78696c%22 "HTTP/1.1 200 OK"
INFO:p-85697:t-8310546624:_client.py:_send_single_request:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/ibm-granite/granite-timeseries-ttm-r2/d6a79570cac0f33d526601cd3a0fc7c80a8f9a2f/config.json?%2Fibm-granite%2Fgranite-timeseries-ttm-r2%2Fresolve%2Fmain%2Fconfig.json=&etag=%2209e937ef5082573c0785a180f23753110b78696c%22 "HTTP/1.1 200 OK"
INFO:p-85697:t-8310546624:_client.py:_send_single_request:HTTP Request: HEAD https

TTM loaded
Context length   : 512
Prediction length: 96


In [20]:
# ============================================================
# TTM SMOKE TEST
# ============================================================

w = windows.iloc[0]

train_end = int(w["train_end"])
test_start = int(w["test_start"])
test_end = int(w["test_end"])

prediction_length = test_end - train_end

test_ids = series_ids[:5]

ttm_contexts = []
ttm_means = []
ttm_stds = []

for sid in test_ids:

    y = (
        p4_data.loc[
            (p4_data["series_id"] == sid)
            & (p4_data["d_int"] <= train_end),
            ["d_int", "sales"],
        ]
        .sort_values("d_int")["sales"]
        .to_numpy(dtype=np.float32)
    )

    y = y[-TTM_CONTEXT:]

    # Scaling parameters calculated from context only
    mean = float(y.mean())
    std = float(y.std())

    if std < 1e-6:
        std = 1.0

    y_scaled = (y - mean) / std

    ttm_contexts.append(y_scaled)
    ttm_means.append(mean)
    ttm_stds.append(std)


past_values = torch.tensor(
    np.stack(ttm_contexts),
    dtype=torch.float32,
).unsqueeze(-1).to(DEVICE)

print("Input shape:", past_values.shape)

Input shape: torch.Size([5, 512, 1])


In [21]:
with torch.inference_mode():

    outputs = ttm(
        past_values=past_values
    )

point_scaled = (
    outputs.prediction_outputs
    .detach()
    .cpu()
    .numpy()
)

print("Raw output shape:", point_scaled.shape)

Raw output shape: (5, 96, 1)


In [22]:
# Remove channel dimension
point_scaled = point_scaled[..., 0]

means = np.asarray(
    ttm_means,
    dtype=np.float32,
)[:, None]

stds = np.asarray(
    ttm_stds,
    dtype=np.float32,
)[:, None]

point_pred = (
    point_scaled * stds + means
)

# Only train_end+1 -> test_end is required
point_pred = point_pred[
    :,
    :prediction_length,
]

# Demand cannot be negative
point_pred = np.clip(
    point_pred,
    0,
    None,
)

print("Required forecast shape:", point_pred.shape)
print(
    "Range:",
    float(point_pred.min()),
    "→",
    float(point_pred.max()),
)

Required forecast shape: (5, 56)
Range: 0.0 → 1.6833503246307373


In [24]:
assert past_values.shape == (
    len(test_ids),
    TTM_CONTEXT,
    1,
)

assert point_pred.shape == (
    len(test_ids),
    prediction_length,
)

assert prediction_length == 56

print("TTM smoke test passed")
print("Context ends at d =", train_end)
print(
    "Forecast covers d =",
    train_end + 1,
    "->",
    test_end,
)
print(
    "Test evaluation covers d =",
    test_start,
    "→",
    test_end,
)

TTM smoke test passed
Context ends at d = 1885
Forecast covers d = 1886 -> 1941
Test evaluation covers d = 1914 → 1941


### Probabilistic wrapper

TTM trả về point forecasts thay vì trực tiếp cung cấp các quantile cần thiết cho
giao thức đánh giá.

Để tạo probabilistic forecasts, mỗi chuỗi được gắn với một phân phối đếm:

- **Negative Binomial** khi lịch sử có over-dispersion (`variance > mean > 0`);
- **Poisson** được sử dụng làm fallback trong các trường hợp còn lại.

Tham số dispersion được ước lượng riêng tại từng rolling origin và chỉ sử dụng
các quan sát `d_int <= train_end`.

Point forecast của TTM được sử dụng làm conditional mean của phân phối. Từ phân
phối này, chín quantile của giao thức đánh giá được tính tương ứng.

In [25]:
# ============================================================
# TTM PROBABILISTIC WRAPPER
# ============================================================

def get_dispersion(data, train_end):
    hist = data.loc[
        data["d_int"] <= train_end,
        ["series_id", "sales"],
    ]

    mv = (
        hist
        .groupby("series_id")["sales"]
        .agg(["mean", "var"])
    )

    mean = mv["mean"].to_numpy()
    var = mv["var"].to_numpy()

    r = np.full(
        len(mv),
        np.inf,
        dtype=np.float64,
    )

    use_nb = (
        (mean > 0)
        & (var > mean)
    )

    r[use_nb] = (
        mean[use_nb] ** 2
        / (var[use_nb] - mean[use_nb])
    )

    return dict(
        zip(
            mv.index,
            r,
        )
    )

In [27]:
def point_to_quantiles(
    point_df,
    model_name,
    disp_r,
):
    
    pf = point_df.copy()

    pf["value"] = (pf["value"].clip(lower=0).astype(float))

    pf["r"] = (pf["series_id"].map(disp_r).fillna(np.inf))

    mu = pf["value"].to_numpy()
    r = pf["r"].to_numpy()

    finite = (np.isfinite(r) & (r > 0))

    outputs = []

    for q in Q:

        qv = np.empty_like(mu, dtype=float,)

        # Negative Binomial
        if finite.any():

            rr = r[finite]
            mm = mu[finite]

            p = rr / (
                rr + np.where(
                    mm > 0,
                    mm,
                    1e-9,
                )
            )

            qv[finite] = stats.nbinom.ppf(
                q,
                rr,
                p,
            )

        # Poisson fallback
        if (~finite).any():

            qv[~finite] = stats.poisson.ppf(
                q,
                mu[~finite],
            )

        block = pf[
            ["series_id", "date", "d_int"]
        ].copy()

        block["quantile_level"] = float(q)

        block["value"] = np.nan_to_num(
            qv,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        block["model"] = model_name

        outputs.append(block)

    result = pd.concat(
        outputs,
        ignore_index=True,
    )

    return result[COLS]

In [28]:
# ============================================================
# PROBABILISTIC WRAPPER SMOKE TEST
# ============================================================

test_offset = test_start - (train_end + 1)

point_test = point_pred[
    :,
    test_offset:test_offset + H,
]

assert point_test.shape == (
    len(test_ids),
    H,
)

test_days = np.arange(
    test_start,
    test_end + 1,
)

point_test_df = pd.DataFrame({
    "series_id": np.repeat(
        test_ids,
        H,
    ),
    "d_int": np.tile(
        test_days,
        len(test_ids),
    ),
    "value": point_test.reshape(-1),
})

point_test_df["date"] = (
    point_test_df["d_int"]
    .map(day_to_date)
)

disp_test = get_dispersion(
    p4_data[
        p4_data["series_id"].isin(test_ids)
    ],
    train_end,
)

ttm_q_test = point_to_quantiles(
    point_test_df,
    "ttm",
    disp_test,
)

print("Point forecast :", point_test_df.shape)
print("Quantile output:", ttm_q_test.shape)

display(ttm_q_test.head(18))

Point forecast : (140, 4)
Quantile output: (1260, 6)


,series_id,date,d_int,quantile_level,value,model
0,FOODS_1_001_CA_2_evaluation,2016-04-25,1914,0.005,0.0,ttm
1,FOODS_1_001_CA_2_evaluation,2016-04-26,1915,0.005,0.0,ttm
2,FOODS_1_001_CA_2_evaluation,2016-04-27,1916,0.005,0.0,ttm
3,FOODS_1_001_CA_2_evaluation,2016-04-28,1917,0.005,0.0,ttm
4,FOODS_1_001_CA_2_evaluation,2016-04-29,1918,0.005,0.0,ttm
5,FOODS_1_001_CA_2_evaluation,2016-04-30,1919,0.005,0.0,ttm
6,FOODS_1_001_CA_2_evaluation,2016-05-01,1920,0.005,0.0,ttm
7,FOODS_1_001_CA_2_evaluation,2016-05-02,1921,0.005,0.0,ttm
8,FOODS_1_001_CA_2_evaluation,2016-05-03,1922,0.005,0.0,ttm
9,FOODS_1_001_CA_2_evaluation,2016-05-04,1923,0.005,0.0,ttm


In [29]:
check_quantile_crossing(ttm_q_test, "ttm",)

assert len(ttm_q_test) == (len(test_ids) * H * len(Q))

assert not ttm_q_test["value"].isna().any()
assert np.isfinite(ttm_q_test["value"]).all()
assert (ttm_q_test["value"] >= 0).all()

print("TTM probabilistic wrapper passed")

✓ ttm: quantile crossing = 0
TTM probabilistic wrapper passed


In [30]:
# ============================================================
# TTM FULL ROLLING INFERENCE
# ============================================================

TTM_BATCH_SIZE = 64

ttm_outputs = []
ttm_runtime = []

print(f"Series     : {len(series_ids):,}")
print(f"Windows    : {len(windows)}")
print(f"Batch size : {TTM_BATCH_SIZE}")
print(f"Device     : {DEVICE}")

Series     : 3,474
Windows    : 3
Batch size : 64
Device     : mps


In [ ]:
for w_idx, w in windows.iterrows():
    window_name = str(w["window"]) if "window" in windows.columns else f"W{w_idx + 1}"

    train_end = int(w["train_end"])
    test_start = int(w["test_start"])
    test_end = int(w["test_end"])
    prediction_length = test_end - train_end
    test_offset = test_start - (train_end + 1)

    print(f"\n{'=' * 60}")
    print(f"{window_name} | train_end={train_end} | test={test_start}–{test_end} | forecast_length={prediction_length}")

    window_start = time.perf_counter()
    window_points = []

    disp_r = get_dispersion(p4_data, train_end)

    for batch_no, batch_start in enumerate(range(0, len(series_ids), TTM_BATCH_SIZE), start=1):
        batch_ids = series_ids[batch_start:batch_start + TTM_BATCH_SIZE]

        contexts, means, stds = [], [], []

        for sid in batch_ids:
            y = p4_data.loc[
                (p4_data["series_id"] == sid) & (p4_data["d_int"] <= train_end),
                ["d_int", "sales"]
            ].sort_values("d_int")["sales"].to_numpy(dtype=np.float32)

            y = y[-TTM_CONTEXT:]

            mean = float(y.mean())
            std = float(y.std())
            if std < 1e-6:
                std = 1.0

            contexts.append((y - mean) / std)
            means.append(mean)
            stds.append(std)

        past_values = torch.tensor(np.stack(contexts), dtype=torch.float32).unsqueeze(-1).to(DEVICE)

        with torch.inference_mode():
            outputs = ttm(past_values=past_values)

        point_scaled = outputs.prediction_outputs.detach().cpu().numpy()[..., 0]

        means = np.asarray(means, dtype=np.float32)[:, None]
        stds = np.asarray(stds, dtype=np.float32)[:, None]

        point_pred = point_scaled * stds + means
        point_pred = np.clip(point_pred[:, :prediction_length], 0, None)

        # Keep test horizon only
        point_test = point_pred[:, test_offset:test_offset + H]
        assert point_test.shape == (len(batch_ids), H)

        test_days = np.arange(test_start, test_end + 1)

        batch_df = pd.DataFrame({
            "series_id": np.repeat(batch_ids, H),
            "d_int": np.tile(test_days, len(batch_ids)),
            "value": point_test.reshape(-1),
        })

        batch_df["date"] = batch_df["d_int"].map(day_to_date)
        window_points.append(batch_df)

        done = min(batch_start + TTM_BATCH_SIZE, len(series_ids))

        if batch_no % 10 == 0 or done == len(series_ids):
            print(f"  Batch {batch_no:02d} | {done:,}/{len(series_ids):,} series")

    # Point forecasts for complete window
    point_window = pd.concat(window_points, ignore_index=True)

    # Point -> probabilistic forecast
    window_df = point_to_quantiles(point_window, "ttm", disp_r)

    elapsed = time.perf_counter() - window_start

    ttm_runtime.append({
        "model": "ttm",
        "window": window_name,
        "n_series": len(series_ids),
        "prediction_length": prediction_length,
        "seconds": elapsed,
    })

    ttm_outputs.append(window_df)

    # Save completed window
    window_path = PATH_OUTPUT / f"ttm_{window_name}.parquet"
    window_df.to_parquet(window_path, index=False)

    print(f"  {window_name}: {len(window_df):,} rows | {elapsed:.1f}s")
    print(f"  Saved: {window_path}")

    del window_points, point_window, window_df


W1 | train_end=1885 | test=1914–1941 | forecast_length=56
  Batch 10 | 640/3,474 series
  Batch 20 | 1,280/3,474 series
  Batch 30 | 1,920/3,474 series
  Batch 40 | 2,560/3,474 series
  Batch 50 | 3,200/3,474 series
  Batch 55 | 3,474/3,474 series
  W1: 875,448 rows | 900.3s
  Saved: /Users/ntt2711/m5-intermittent-forecasting/data/outputs/forecasts/ttm_W1.parquet

W2 | train_end=1857 | test=1886–1913 | forecast_length=56
  Batch 10 | 640/3,474 series
  Batch 20 | 1,280/3,474 series
  Batch 30 | 1,920/3,474 series
  Batch 40 | 2,560/3,474 series
  Batch 50 | 3,200/3,474 series
  Batch 55 | 3,474/3,474 series
  W2: 875,448 rows | 892.7s
  Saved: /Users/ntt2711/m5-intermittent-forecasting/data/outputs/forecasts/ttm_W2.parquet

W3 | train_end=1829 | test=1858–1885 | forecast_length=56
  Batch 10 | 640/3,474 series
  Batch 20 | 1,280/3,474 series
  Batch 30 | 1,920/3,474 series
  Batch 40 | 2,560/3,474 series
  Batch 50 | 3,200/3,474 series


In [ ]:
# ============================================================
# COMBINE & VALIDATE TTM FORECAST
# ============================================================

ttm_all = pd.concat(ttm_outputs, ignore_index=True)

ttm_all["quantile_level"] = ttm_all["quantile_level"].astype(float)
ttm_all["value"] = ttm_all["value"].astype(np.float32)

validate_forecast(ttm_all, "ttm")
check_quantile_crossing(ttm_all, "ttm")

ttm_runtime_df = pd.DataFrame(ttm_runtime)
display(ttm_runtime_df)

print(f"Total TTM runtime: {ttm_runtime_df['seconds'].sum():.1f}s")
print(f"AVG per window: {ttm_runtime_df['seconds'].mean():.1f}s")

## Cell cuối — lưu output (điền biến theo comment rồi chạy)

In [ ]:
# ===== lưu output (mỗi model 1 file, ĐÚNG 6 cột) =====
# forecasts = {"naive": df, "snaive": df, "croston": df, "sba": df, "tsb": df, "chronos_bolt": df, "ttm": df}
COLS = ["series_id","date","d_int","quantile_level","value","model"]
Path("data/outputs/forecasts").mkdir(parents=True, exist_ok=True)
for name, df in forecasts.items():
    assert set(COLS).issubset(df.columns), f"{name} thiếu cột! cần đủ {COLS}"
    df[COLS].to_parquet(f"data/outputs/forecasts/{name}.parquet", index=False); print("✅ lưu", name, df.shape)